In [1]:
import pandas as pd
import numpy as np

In [2]:
matches = pd.read_csv("/Users/ishagarg/Documents/PythonProjects/ipl-player-intelligence/data/matches.csv")
deliveries = pd.read_csv("/Users/ishagarg/Documents/PythonProjects/ipl-player-intelligence/data/deliveries.csv")

In [3]:
print("Matches shape:", matches.shape)
print("Deliveries shape:", deliveries.shape)

Matches shape: (1095, 20)
Deliveries shape: (260920, 17)


In [ ]:
# See all column names
print("MATCHES columns:", matches.columns.tolist())
print("\nDELIVERIES columns:", deliveries.columns.tolist())

In [ ]:
print("Matches missing values:")
print(matches.isnull().sum())

print("\nDeliveries missing values:")
print(deliveries.isnull().sum())

In [ ]:
matches.head()
deliveries.head()

In [ ]:
#Batsman Analysis

In [5]:
matches["season"].unique()

<ArrowStringArray>
['2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016',
 '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Length: 17, dtype: str

In [7]:
matches["season"] = matches["season"].replace({'2007/08':'2008','2009/10':'2010','2020/21':'2020'}, inplace = True)
matches["season"].unique()

/var/folders/z0/xmfn206935g7qmm7d674mbz00000gn/T/ipykernel_13380/931372263.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  matches["season"] = matches["season"].replace({'2007/08':'2008','2009/10':'2010','2020/21':'2020'}, inplace = True)


<ArrowStringArray>
['2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016',
 '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Length: 17, dtype: str

In [8]:
print("Total numbers of players who bowled: ",
deliveries["bowler"].nunique())

Total numbers of players who bowled:  530


In [9]:
print("Total numbers of players who batted: ",
deliveries["batter"].nunique())

Total numbers of players who batted:  673


In [10]:
top_scorers = (deliveries
    .groupby('batter')['batsman_runs']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index())

top_scorers

,batter,batsman_runs
0,V Kohli,8014
1,S Dhawan,6769
2,RG Sharma,6630
3,DA Warner,6567
4,SK Raina,5536
5,MS Dhoni,5243
6,AB de Villiers,5181
7,CH Gayle,4997
8,RV Uthappa,4954
9,KD Karthik,4843


In [11]:
top_scorers.columns = ['Player', 'Total Runs']
print(top_scorers)

           Player  Total Runs
0         V Kohli        8014
1        S Dhawan        6769
2       RG Sharma        6630
3       DA Warner        6567
4        SK Raina        5536
5        MS Dhoni        5243
6  AB de Villiers        5181
7        CH Gayle        4997
8      RV Uthappa        4954
9      KD Karthik        4843


In [12]:
# --- Bowler Stats ---

In [13]:
top_bowlers = deliveries.groupby('bowler')['is_wicket'].sum().sort_values(ascending = False).head(10).reset_index()

In [14]:
top_bowlers

,bowler,is_wicket
0,YS Chahal,213
1,DJ Bravo,207
2,PP Chawla,201
3,SP Narine,200
4,R Ashwin,198
5,B Kumar,195
6,SL Malinga,188
7,A Mishra,183
8,JJ Bumrah,182
9,RA Jadeja,169


In [15]:
balls_faced = deliveries.groupby('batter').size()
print(balls_faced.head(10))

runs_scored = deliveries.groupby('batter')['batsman_runs'].sum()
print("\n",runs_scored.head(10))

strike_rate = (runs_scored / balls_faced * 100).round(2)
# strike_rate.shape
# 673
qualified = strike_rate[balls_faced >= 500].sort_values(ascending=False).head(10)

qualified

batter
A Ashish Reddy    196
A Badoni          505
A Chandila          7
A Chopra           75
A Choudhary        20
A Dananjaya         5
A Flintoff         57
A Kamboj            2
A Kumble           49
A Manohar         181
dtype: int64

 batter
A Ashish Reddy    280
A Badoni          634
A Chandila          4
A Chopra           53
A Choudhary        25
A Dananjaya         4
A Flintoff         62
A Kamboj            2
A Kumble           35
A Manohar         231
Name: batsman_runs, dtype: int64


batter
AD Russell         164.22
H Klaasen          161.99
SP Narine          155.89
N Pooran           154.77
LS Livingstone     154.19
GJ Maxwell         150.49
RM Patidar         149.63
Abhishek Sharma    148.86
V Sehwag           148.83
AB de Villiers     148.58
dtype: float64

In [16]:
# Total runs conceded per bowler
bowler_runs = deliveries.groupby('bowler')['total_runs'].sum()

# Total balls bowled (exclude wides and no balls)
legal_balls = deliveries[~deliveries['extras_type'].isin(['wides', 'noballs'])]
bowler_balls = legal_balls.groupby('bowler').size()

# Overs bowled
bowler_overs = (bowler_balls / 6).round(2)

# Economy rate
economy = (bowler_runs / bowler_overs).round(2)

# Wickets taken
wickets = deliveries[deliveries['is_wicket'] == 1]\
    .groupby('bowler').size()

# Average (runs per wicket)
bowling_avg = (bowler_runs / wickets).round(2)

# Strike rate (balls per wicket)
bowling_sr = (bowler_balls / wickets).round(2)

# Combine everything
bowler_summary = pd.DataFrame({
    'bowler': bowler_runs.index,
    'total_runs_conceded': bowler_runs.values,
    'balls_bowled': bowler_balls.reindex(bowler_runs.index).values,
    'overs_bowled': bowler_overs.reindex(bowler_runs.index).values,
    'economy': economy.reindex(bowler_runs.index).values,
    'wickets': wickets.reindex(bowler_runs.index).fillna(0).values,
    'bowling_avg': bowling_avg.reindex(bowler_runs.index).values,
    'bowling_sr': bowling_sr.reindex(bowler_runs.index).values,
}).dropna(subset=['balls_bowled'])

# Filter meaningful data — minimum 120 balls (20 overs)
bowler_summary = bowler_summary[bowler_summary['balls_bowled'] >= 120]\
    .sort_values('wickets', ascending=False)\
    .reset_index(drop=True)

print(f"Total bowlers: {len(bowler_summary)}")
print(bowler_summary.head(10))

Total bowlers: 318
       bowler  total_runs_conceded  balls_bowled  overs_bowled  economy  \
0   YS Chahal                 4681          3521        586.83     7.98   
1    DJ Bravo                 4436          3120        520.00     8.53   
2   PP Chawla                 5179          3850        641.67     8.07   
3   SP Narine                 4672          4081        680.17     6.87   
4    R Ashwin                 5435          4524        754.00     7.21   
5     B Kumar                 5051          3910        651.67     7.75   
6  SL Malinga                 3486          2828        471.33     7.40   
7    A Mishra                 4193          3371        561.83     7.46   
8   JJ Bumrah                 3840          3075        512.50     7.49   
9   RA Jadeja                 4917          3829        638.17     7.70   

   wickets  bowling_avg  bowling_sr  
0    213.0        21.98       16.53  
1    207.0        21.43       15.07  
2    201.0        25.77       19.15  
3  

In [17]:
bowler_runs = deliveries.groupby('bowler')['total_runs'].sum()
bowler_balls = deliveries.groupby('bowler').size()
bowler_overs = bowler_balls / 6

economy = (bowler_runs / bowler_overs).round(2)
qualified_bowlers = economy[bowler_overs >= 200].sort_values().head(10)

print(qualified_bowlers)

bowler
M Muralitharan     6.70
SP Narine          6.76
DW Steyn           6.79
Rashid Khan        6.91
R Ashwin           6.97
SL Malinga         7.03
Harbhajan Singh    7.04
SK Warne           7.19
JJ Bumrah          7.23
KH Pandya          7.27
dtype: float64


In [18]:
# Create a player summary table — to load this into Supabase
batsman_summary = pd.DataFrame({
    'player': runs_scored.index,
    'total_runs': runs_scored.values,
    'balls_faced': balls_faced.reindex(runs_scored.index).values,
    'strike_rate': strike_rate.reindex(runs_scored.index).round(2).values
}).dropna().sort_values('total_runs', ascending=False)

batsman_summary.to_csv('/Users/ishagarg/Documents/PythonProjects/ipl-player-intelligence/data/batsman_summary.csv', index=False)
print("Saved! Rows:", len(batsman_summary))

Saved! Rows: 673


In [19]:
##### Supabase Upload

In [20]:
import os
from supabase import create_client
from dotenv import load_dotenv
import pandas as pd

load_dotenv("/Users/ishagarg/Documents/PythonProjects/ipl-player-intelligence/.env")

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

supabase = create_client(url, key)

print(os.getcwd())
# oai_key = os.getenv("OPENAI_API_KEY")
# print(f"Key loaded: {oai_key}..." if oai_key else "❌ Key not found")

# # Print to verify credentials loaded
# print("URL:", os.getenv("SUPABASE_URL"))
# print("Key:", os.getenv("SUPABASE_KEY")[:8], "...")

# Now test query
result = supabase.table('batsman_summary').select("*").limit(1).execute()
print(result.data)

def query_supabase(table, filters=None):
    """
    Simple query function - name of the table to query
    filters: optional dict of column:value filters
    """
    query = supabase.table(table).select("*")
    
    if filters:
        for column, value in filters.items():
            query = query.eq(column, value)
    
    result = query.execute()
    return pd.DataFrame(result.data)

# Test it
df = query_supabase('matches')
print(f"✅ Got {len(df)} rows from matches table")
print(df.head())



/Users/ishagarg/Documents/PythonProjects/ipl-player-intelligence/notebooks
[]
✅ Got 0 rows from matches table
Empty DataFrame
Columns: []
Index: []


In [21]:
# Load your CSVs
matches = pd.read_csv('../data/matches.csv')
deliveries = pd.read_csv('../data/deliveries.csv')
batsman_summary = pd.read_csv('../data/batsman_summary.csv')

# Clean column names — Supabase doesn't like spaces
matches.columns = matches.columns.str.lower().str.replace(' ', '_')
deliveries.columns = deliveries.columns.str.lower().str.replace(' ', '_')
matches["season"] = matches["season"].replace({'2007/08':'2008','2009/10':'2010','2020/21':'2020'}, inplace = True)

print("Ready to upload")
print(f"Matches: {len(matches)} rows")
print(f"Deliveries: {len(deliveries)} rows")
print(f"Player summary: {len(batsman_summary)} rows")

Ready to upload
Matches: 1095 rows
Deliveries: 260920 rows
Player summary: 673 rows


/var/folders/z0/xmfn206935g7qmm7d674mbz00000gn/T/ipykernel_13380/1787173876.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  matches["season"] = matches["season"].replace({'2007/08':'2008','2009/10':'2010','2020/21':'2020'}, inplace = True)


In [22]:
matches["season"].unique()

<ArrowStringArray>
['2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016',
 '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Length: 17, dtype: str

In [29]:
# Truncating tables if exist
response = supabase.table('matches').delete().neq('id', 0).execute()
response = supabase.table('deliveries').delete().neq('id', 0).execute()
response = supabase.table('batsman_summary').delete().neq('id', 0).execute()
response = supabase.table('bowler_summary').delete().neq('id', 0).execute()

APIError: {'message': 'column bowler_summary.id does not exist', 'code': '42703', 'hint': None, 'details': None}

In [24]:
# Upload in batches — Supabase has row limits per request
def upload_in_batches(supabase, table_name, df, batch_size=500):
    # df = df.where(pd.notnull(df), None)  # replace NaN with None
    df = df.replace({np.nan: None})
    records = df.to_dict('records')
    total = len(records)
    
    for i in range(0, total, batch_size):
        batch = records[i:i+batch_size]
        supabase.table(table_name).insert(batch).execute()
        print(f"Uploaded {min(i+batch_size, total)}/{total} rows")
    
    print(f"✓ {table_name} upload complete")

# Upload matches first (smaller table)
upload_in_batches(supabase, 'matches', matches)

Uploaded 500/1095 rows
Uploaded 1000/1095 rows
Uploaded 1095/1095 rows
✓ matches upload complete


In [25]:
upload_in_batches(supabase, 'batsman_summary', batsman_summary)

Uploaded 500/673 rows
Uploaded 673/673 rows
✓ batsman_summary upload complete


In [31]:
upload_in_batches(supabase, 'bowler_summary', bowler_summary)

Uploaded 318/318 rows
✓ bowler_summary upload complete


In [27]:
# OpenAI API Calls

In [28]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "Say hello in 5 words"}
    ]
)
print(response.choices[0].message.content)

Greetings and salutations, howdy friend!


In [ ]:
def ask_ipl(schema, question):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": schema},
            {"role": "user", "content": question}
        ]
    )
    
    return response.choices[0].message.content

# Test it
# Tell the AI about your database schema
schema = """
You have access to these tables:
    
batsman_summary: player(name), total_runs, balls_faced, strike_rate
matches: id, season, city, date, team1, team2, winner, player_of_match
    
Write a PostgreSQL SQL query to answer the user's question.
Return ONLY the SQL query, nothing else.
"""
question = "Who are the top 5 players by total runs?"
sql = ask_ipl(schema, question)
print("Generated SQL:")
print(sql)


In [33]:
# Get top 20 players from your database
result = supabase.table('batsman_summary')\
    .select("*")\
    .order('total_runs', desc=True)\
    .limit(20)\
    .execute()

players_df = pd.DataFrame(result.data)
print(players_df)

            player  total_runs  balls_faced  strike_rate
0          V Kohli        8014         6236       128.51
1         S Dhawan        6769         5483       123.45
2        RG Sharma        6630         5183       127.92
3        DA Warner        6567         4849       135.43
4         SK Raina        5536         4177       132.54
5         MS Dhoni        5243         3947       132.84
6   AB de Villiers        5181         3487       148.58
7         CH Gayle        4997         3516       142.12
8       RV Uthappa        4954         3927       126.15
9       KD Karthik        4843         3687       131.35
10        KL Rahul        4689         3578       131.05
11       AM Rahane        4642         3858       120.32
12    F du Plessis        4571         3435       133.07
13       SV Samson        4419         3270       135.14
14       AT Rayudu        4348         3490       124.58
15       G Gambhir        4217         3524       119.67
16       SR Watson        3880 

In [35]:
result = supabase.table('bowler_summary')\
    .select("*")\
    .order('wickets', desc=True)\
    .limit(20)\
    .execute()

bowlers_df = pd.DataFrame(result.data)
print(bowlers_df)

             bowler  total_runs_conceded  balls_bowled  overs_bowled  economy  \
0         YS Chahal                 4681          3521        586.83     7.98   
1          DJ Bravo                 4436          3120        520.00     8.53   
2         PP Chawla                 5179          3850        641.67     8.07   
3         SP Narine                 4672          4081        680.17     6.87   
4          R Ashwin                 5435          4524        754.00     7.21   
5           B Kumar                 5051          3910        651.67     7.75   
6        SL Malinga                 3486          2828        471.33     7.40   
7          A Mishra                 4193          3371        561.83     7.46   
8         JJ Bumrah                 3840          3075        512.50     7.49   
9         RA Jadeja                 4917          3829        638.17     7.70   
10         UT Yadav                 4442          3057        509.50     8.72   
11  Harbhajan Singh         

In [36]:
# Convert player data to readable text for the AI
players_text = players_df.to_string(index=False)
bowlers_text = bowlers_df.to_string(index=False)

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {
            "role": "system",
            "content": """You are an expert IPL cricket analyst with deep knowledge 
            of player performance, batting techniques, and match situations. 
            Give specific, insight-driven analysis — not generic commentary."""
        },
        {
            "role": "user",
            "content": f"""Here is IPL player performance data:

{bowlers_text}

Question: Who is the most economical bowler?"""
        }
    ]
)

print(response.choices[0].message.content)

To determine the most economical bowler from the provided IPL player performance data, we need to look at the "economy" stat for each bowler. The lower the economy rate, the more economical a bowler is considered.

From the data provided:

1. YS Chahal - Economy: 7.98
2. DJ Bravo - Economy: 8.53
3. PP Chawla - Economy: 8.07
4. SP Narine - Economy: 6.87
5. R Ashwin - Economy: 7.21
6. B Kumar - Economy: 7.75
7. SL Malinga - Economy: 7.40
8. A Mishra - Economy: 7.46
9. JJ Bumrah - Economy: 7.49
10. RA Jadeja - Economy: 7.70
11. UT Yadav - Economy: 8.72
12. Harbhajan Singh - Economy: 7.20
13. Rashid Khan - Economy: 6.98
14. Sandeep Sharma - Economy: 7.99
15. HV Patel - Economy: 8.92
16. MM Sharma - Economy: 8.73
17. Mohammed Shami - Economy: 8.59
18. AR Patel - Economy: 7.39
19. TA Boult - Economy: 8.42
20. R Vinay Kumar - Economy: 8.58

Based on the economy rates provided, Sunil Narine has the lowest economy rate of 6.87, making him the most economical bowler among the listed players. His

In [ ]:
def generate_sql(question):
    schema = """
    You have access to these PostgreSQL tables:
    
    batsman_summary:
    - player (TEXT): player name
    - total_runs (FLOAT): total runs scored in IPL career
    - balls_faced (FLOAT): total balls faced
    - strike_rate (FLOAT): career strike rate
    
    matches:
    - id (INTEGER): match id
    - season (INTEGER): IPL season year
    - city (TEXT): city where match was played
    - date (TEXT): match date
    - team1 (TEXT): first team
    - team2 (TEXT): second team
    - winner (TEXT): winning team
    - player_of_match (TEXT): player of the match
    - toss_winner (TEXT): toss winning team
    - toss_decision (TEXT): bat or field
    - result (TEXT): match result type
    - result_margin (FLOAT): margin of victory
    - venue (TEXT): stadium name
    
    Rules:
    - Return ONLY the SQL query, no explanation, no markdown
    - Use lowercase column names
    - Always use LIMIT 20 unless asked for specific number
    - For player names use ILIKE for case-insensitive matching
    """
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": schema},
            {"role": "user", "content": question}
        ]
    )
    
    return response.choices[0].message.content.strip()

# Test it
question = "Who are the top 5 players by strike rate with more than 1000 balls faced?"
sql = generate_sql(question)
print("Generated SQL:")
print(sql)

In [ ]:
def run_query(question):
    # Step 1: generate SQL from question
    sql = generate_sql(question)
    print(f"Generated SQL:\n{sql}\n")
    
    # Step 2: clean any markdown the AI might add
    import re
    sql = re.sub(r'```sql|```', '', sql).strip()
    
    # Step 3: run against Supabase using postgrest syntax
    # For now we'll use pandas as our query engine against local data
    # Full SQL execution via Supabase comes in Day 4
    print("Running query...")
    
    # Step 4: generate AI insight from results
    result = supabase.table('batsman_summary')\
        .select("*")\
        .order('strike_rate', desc=True)\
        .limit(5)\
        .execute()
    
    df = pd.DataFrame(result.data)
    
    # Step 5: AI summarizes the result
    summary_response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are an IPL analyst. Summarize data insights in 3 sentences max. Be specific with numbers."},
            {"role": "user", "content": f"Question: {question}\n\nData:\n{df.to_string(index=False)}\n\nProvide a sharp insight."}
        ]
    )
    
    print("Result:")
    print(df)
    print("\nAI Insight:")
    print(summary_response.choices[0].message.content)

# Run it
run_query("Who are the top 5 players by strike rate with more than 1000 balls faced?")

In [ ]:
from openai import OpenAI
from supabase import create_client
import os
import pandas as pd

# Clients
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# Pull real data from Supabase
result = supabase.table('batsman_summary')\
    .select("*")\
    .order('total_runs', desc=True)\
    .limit(10)\
    .execute()

players_df = pd.DataFrame(result.data)

# Send to AI
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {
            "role": "system",
            "content": "You are an expert IPL cricket analyst. Give specific, insight-driven analysis."
        },
        {
            "role": "user",
            "content": f"""Here is IPL player performance data:

{players_df.to_string(index=False)}

Who are the top 3 most valuable batsmen and why? Consider both total runs and strike rate."""
        }
    ]
)

print(response.choices[0].message.content)